# 前置作業

In [77]:
#!pip install google.colab #如未安裝取消註解後執行
import os

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [78]:
import matplotlib as mpl
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt

# 下載繁體中文字型
!wget -O SourceHanSerifTW-VF.ttf https://github.com/adobe-fonts/source-han-serif/raw/release/Variable/TTF/Subset/SourceHanSerifTW-VF.ttf

# 加入字型檔
fm.fontManager.addfont('SourceHanSerifTW-VF.ttf')

# 設定字型
#
mpl.rc('font', family='Source Han Serif TW VF')

--2026-05-22 07:14:09--  https://github.com/adobe-fonts/source-han-serif/raw/release/Variable/TTF/Subset/SourceHanSerifTW-VF.ttf
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/adobe-fonts/source-han-serif/release/Variable/TTF/Subset/SourceHanSerifTW-VF.ttf [following]
--2026-05-22 07:14:09--  https://raw.githubusercontent.com/adobe-fonts/source-han-serif/release/Variable/TTF/Subset/SourceHanSerifTW-VF.ttf
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 16851180 (16M) [application/octet-stream]
Saving to: ‘SourceHanSerifTW-VF.ttf’

SourceHanSerifTW-VF 100%[===================>]  16.07M  --.-KB/s

In [79]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

import os
import base64
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from PIL import Image

# 設定繪圖包括中文字型
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pandas.plotting import scatter_matrix

# 程式拆段

## Step1: 策略庫 (Strategies Pool)

我們可以把策略拆解成兩個動作：

buy_strategy：輸入當月資料，決定要「扣款多少錢」。

sell_strategy：輸入當月資料，決定要「賣出多少比例或股數」。

In [80]:
# ==========================================
# 1. 策略庫 (Strategies Pool)
# ==========================================

# --- 策略一：一次單筆 All-in 策略 ---
def lump_sum_all_in_buy(row, state):
    """
    第一個月直接將所有可用現金全部投入，之後月份皆不投入。
    """
    # 如果 state 沒給預設值，預設為 True
    if state.get("is_first_month", True):
        state["is_first_month"] = False
        return state["cycle_start_cash"]  # 全部現金作為預算
    return 0


# --- 策略二：純「定期定額法」 ---
def regular_fixed_buy(row, state):
    """每個月雷打不動固定投入 10,000 元"""
    return 10000


# --- 策略三：不定期不定額（越跌買越多） ---
def irregular_dynamic_buy(row, state):
    """依據景氣低迷程度加碼：藍燈扣2萬、黃藍燈1.5萬、綠燈1萬，高檔不扣款"""
    lamp = row['lamp_name']
    if lamp == "藍燈":
        return 20000
    elif lamp == "黃藍燈":
        return 15000
    elif lamp == "綠燈":
        return 10000
    else:
        return 0  # 黃紅燈、紅燈高檔時停止扣款，留存現金


# --- 策略四：彼得林區+阿甘（定期定額 + 不定期不定額加碼） ---
def peter_lynch_dynamic_buy(row, state):
    """
    基本每月定期定額 10,000 元，
    若遇到低迷景氣則「外加」加碼額度：藍燈再加2萬、黃藍燈再加1.5萬、綠燈再加1萬。
    """
    lamp = row['lamp_name']
    base_amount = 10000

    if lamp == "藍燈":
        return base_amount + 20000
    elif lamp == "黃藍燈":
        return base_amount + 15000
    elif lamp == "綠燈":
        return base_amount + 10000
    else:
        return base_amount


# --- 策略五：景氣燈號分批進出場策略 ---
def lamp_batch_buy(row, state):
    """景氣燈號進場邏輯：首次遇到買進燈號分批買，綠燈暫停"""
    lamp = row['lamp_name']


    # 綠燈：暫停分批買進
    if lamp == "綠燈" and (state["buy_active"] or state["buy_count"] > 0):
        state["buy_active"] = False

    # 如果已經觸發賣出序列，本循環不再買進
    if state["sell_active"]:
        return 0

    # 檢查是否觸發買進訊號(is_buy_signal) (藍燈或黃藍燈)
    lamp_rank = {"藍燈": 1, "黃藍燈": 2, "綠燈": 3, "黃紅燈": 4, "紅燈": 5}
    is_buy_signal = lamp_rank.get(lamp, 999) <= lamp_rank.get(state["buy_lamp_input"], 2)

    if state["buy_count"] < state["max_buy"] and (state["buy_active"] or is_buy_signal):
        state["buy_active"] = True
        buy_budget_per_time = state["cycle_start_cash"] / state["max_buy"]
        state["buy_count"] += 1

        if state["buy_count"] >= state["max_buy"]:
            state["buy_active"] = False

        return buy_budget_per_time

    return 0

def lamp_batch_sell(row, state, current_shares):
    """景氣燈號出場邏輯：遇到賣出燈號分批賣，賣光則重置循環"""
    lamp = row['lamp_name']


    # 檢查是否觸發賣出訊號(is_sell_signal) (黃紅燈或紅燈)
    lamp_rank = {"藍燈": 1, "黃藍燈": 2, "綠燈": 3, "黃紅燈": 4, "紅燈": 5}
    is_sell_signal = lamp_rank.get(lamp, 0) >= lamp_rank.get(state["sell_lamp_input"], 4)

    if state["sell_count"] < state["max_sell"] and current_shares > 0 and (state["sell_active"] or is_sell_signal):
        state["buy_active"] = False  # 出場優先
        state["sell_active"] = True

        portions_left = state["max_sell"] - state["sell_count"]
        if portions_left == 1:
            sell_shares = current_shares
        else:
            sell_shares = max(1, int(current_shares // portions_left))

        sell_shares = min(sell_shares, current_shares)
        state["sell_count"] += 1

        return sell_shares

    return 0


# --- 通用不賣出函數 ---
def never_sell(row, state, current_shares):
    return 0

## Step2: 高彈性回測核心引擎 (Backtest Engine)

這個引擎只負責管理「錢包（現金）」和「股票（庫存）」，並記錄歷史。它不關心現在用的是什麼策略，而是像外包一樣，每個月去詢問你指定的策略函數：「老闆，我這個月要買多少、賣多少？」

In [81]:
# ==========================================
# 2. 高彈性回測核心引擎 (Backtest Engine)
# ==========================================
def backtest_engine(df, initial_capital, buy_strategy_func, sell_strategy_func, strategy_params=None):
    """
    通用回測引擎：完全與特定策略解耦，僅維護基礎現金與持股
    """
    cash = initial_capital
    shares = 0

    # 引擎基礎狀態：只管每一輪循環開始時的資金水位
    state = {
        "cycle_start_cash": initial_capital
    }

    # 融合外部傳入的策略專屬參數與初始狀態
    if strategy_params:
        state.update(strategy_params)

    history = []
    total_records = len(df)

    for i in range(total_records):
        row = df.iloc[i]
        has_next = (i + 1 < total_records)
        next_open = df.iloc[i + 1]['open_price'] if has_next else row['open_price']

        action = "觀望"

        # 1. 執行賣出策略判定
        sell_shares_target = sell_strategy_func(row, state, shares)
        if sell_shares_target > 0 and shares > 0:
            actual_sell = min(sell_shares_target, shares)
            cash += actual_sell * next_open
            shares -= actual_sell
            action = f"賣出 {actual_sell} 股"

            # 若股票全部賣光（循環結束），進行結算重置
            if shares == 0:
                # 基礎重置
                state["cycle_start_cash"] = cash

                # 動態重置特定策略的內部計數狀態（如果策略有使用到的話欄位）
                if "buy_count" in state: state["buy_count"] = 0
                if "sell_count" in state: state["sell_count"] = 0
                if "buy_active" in state: state["buy_active"] = False
                if "sell_active" in state: state["sell_active"] = False

        # 2. 執行買進策略判定
        elif has_next:
            buy_budget = buy_strategy_func(row, state)
            if buy_budget > 0 and cash > 0:
                spend = min(cash, buy_budget)
                bought_shares = int(spend // next_open)
                if bought_shares > 0:
                    cash -= (bought_shares * next_open)
                    shares += bought_shares
                    action = f"買進 {bought_shares} 股"

        # 3. 計算當月總資產
        mark_price = next_open if has_next else row['open_price']
        current_asset = cash + (shares * mark_price)

        history.append({
            "年月": row['ym_format'],
            "景氣分數": row['score'],
            "燈號": row['lamp_name'],
            "0050股價": row['open_price'],
            "執行動作": action,
            "現金水位": cash,
            "持股水位(股)": shares,
            "當下總資產": current_asset
        })

    return pd.DataFrame(history)

# Step3:寫入函數

In [82]:
def save_to_excel(df, sheet_name, filename='執行結果.xlsx'):
    """自動判斷新建或追加 Sheet 的通用寫入函數"""
    # 檢查檔案是否存在，決定使用追加(a)還是新建(w)模式
    mode = 'a' if os.path.exists(filename) else 'w'

    # 只有在追加模式(a)時，才需要傳入 if_sheet_exists 參數
    kwargs = {'if_sheet_exists': 'replace'} if mode == 'a' else {}

    with pd.ExcelWriter(filename, engine='openpyxl', mode=mode, **kwargs) as writer:
        df.to_excel(writer, sheet_name=sheet_name, index=False)

# 寫入函數-寬度調整

In [83]:
import os
import pandas as pd

def save_to_excel(df, sheet_name, filename='執行結果.xlsx'):
    """自動判斷新建/追加，並自動調整欄寬的寫入函數"""
    mode = 'a' if os.path.exists(filename) else 'w'
    kwargs = {'if_sheet_exists': 'replace'} if mode == 'a' else {}

    with pd.ExcelWriter(filename, engine='openpyxl', mode=mode, **kwargs) as writer:
        df.to_excel(writer, sheet_name=sheet_name, index=False)

        # --- 以下為自動調整欄寬邏輯 ---
        # 取得當前寫入的 openpyxl 工作表物件
        worksheet = writer.sheets[sheet_name]

        for col in worksheet.columns:
            # 找出該欄中所有儲存格轉成字串後的最大長度
            max_len = 0
            col_letter = col[0].column_letter # 取得欄位英文字母 (如 'A', 'B')

            for cell in col:
                if cell.value is not None:
                    # 處理中文字元長度（中文字在 Excel 佔 2 個字元寬度）
                    val_str = str(cell.value)
                    # 計算實際顯示長度：英文/數字算 1，中文字算 2
                    byte_len = len(val_str.encode('utf-8-sig'))
                    # utf-8 編碼下中文字通常是 3 寬度，這裡取折衷或直接用簡單長度
                    # 實務上最安全且簡單的中文長度估算：
                    display_len = sum(2 if ord(char) > 127 else 1 for char in val_str)

                    if display_len > max_len:
                        max_len = display_len

            # 設定欄寬（最大字數 + 4 格安全緩衝，避免邊緣切到）
            # 設定寬度最小不低於 12，避免標題太短時欄位過窄
            worksheet.column_dimensions[col_letter].width = max(max_len + 4, 12)

# Step4: 主程式-資料載入與回測執行

In [84]:
basepath = "/content/drive/MyDrive/[銀行&&券商]/書本、影音/投資小白_2026-05-16~17/分組/"
fileName = "分組_景氣燈號.csv"
lightpath = basepath + "light_img/"

In [85]:
# A. 載入並預處理歷史資料
df_raw = pd.read_csv(basepath + fileName)
df_raw

,ym,score,50
0,2003/6/1,20.0,37.10
1,2003/7/1,24.0,37.09
2,2003/8/1,26.0,41.28
3,2003/9/1,29.0,46.40
4,2003/10/1,31.0,44.51
...,...,...,...
270,2025/12/1,38.0,247.20
271,2026/1/1,39.0,264.00
272,2026/2/1,40.0,286.20
273,2026/3/1,39.0,318.00


In [86]:

    # 建立模擬數據 (若實際執行請改回您的 pd.read_csv("分組_景氣燈號.csv"))
    # date_range = pd.date_range(start="2023-01-01", periods=30, freq="ME")
    # np.random.seed(42)
    # df_raw = pd.DataFrame({
    #     'ym': date_range.strftime('%Y-%m-%d'),
    #     'score': np.random.randint(10, 40, size=30),
    #     '50': np.random.uniform(110, 150, size=30)
    # })

    # A. 資料預處理
    df_raw['ym_dt'] = pd.to_datetime(df_raw['ym'])
    df_raw['ym_format'] = df_raw['ym_dt'].dt.strftime('%Y-%m')
    df_raw = df_raw.rename(columns={'50': 'open_price'})

    # 分數轉燈號
    bins = [8.5, 16.5, 22.5, 31.5, 37.5, 45.5]
    labels = ['藍燈', '黃藍燈', '綠燈', '黃紅燈', '紅燈']
    df_raw['lamp_name'] = pd.cut(df_raw['score'], bins=bins, labels=labels).astype(str).replace('nan', '-')

    # 篩選回測時間範圍
    start_ym, end_ym = "2003-06", "2026-04"
    df_filtered = df_raw[(df_raw['ym_format'] >= start_ym) & (df_raw['ym_format'] <= end_ym)].copy()
    df_filtered = df_filtered.sort_values('ym_dt').reset_index(drop=True)

    initial_money = 1000000  # 初始資金 100 萬

    df_filtered

,ym,score,open_price,ym_dt,ym_format,lamp_name
0,2003/6/1,20.0,37.10,2003-06-01,2003-06,黃藍燈
1,2003/7/1,24.0,37.09,2003-07-01,2003-07,綠燈
2,2003/8/1,26.0,41.28,2003-08-01,2003-08,綠燈
3,2003/9/1,29.0,46.40,2003-09-01,2003-09,綠燈
4,2003/10/1,31.0,44.51,2003-10-01,2003-10,綠燈
...,...,...,...,...,...,...
270,2025/12/1,38.0,247.20,2025-12-01,2025-12,紅燈
271,2026/1/1,39.0,264.00,2026-01-01,2026-01,紅燈
272,2026/2/1,40.0,286.20,2026-02-01,2026-02,紅燈
273,2026/3/1,39.0,318.00,2026-03-01,2026-03,紅燈


## 策略1: 單筆 All-In (期初全下/長期持有)

In [87]:
# --------------------------------------------------
# 策略一：單筆All-In (期初全下/長期持有)
# --------------------------------------------------
all_in_params = {"is_first_month": True}  # 專屬狀態
df_res_all_in = backtest_engine(
    df=df_filtered,
    initial_capital=initial_money,
    buy_strategy_func=lump_sum_all_in_buy,
    sell_strategy_func=never_sell,
    strategy_params=all_in_params
)
asset_all_in = df_res_all_in['當下總資產'].iloc[-1]
roi_all_in = ((asset_all_in - initial_money) / initial_money) * 100

print("====== 策略一：單筆All-in ======")
print(f"期末總金額: NT$ {round(asset_all_in):,}")
print(f"回測報酬率: {roi_all_in:+.2f}%\n")


save_to_excel(df_res_all_in, '1_單筆All-in')

df_res_all_in

====== 策略一：單筆All-in ======
期末總金額: NT$ 8,088,317
回測報酬率: +708.83%



,年月,景氣分數,燈號,0050股價,執行動作,現金水位,持股水位(股),當下總資產
0,2003-06,20.0,黃藍燈,37.10,買進 26961 股,16.51,26961,1000000.00
1,2003-07,24.0,綠燈,37.09,觀望,16.51,26961,1112966.59
2,2003-08,26.0,綠燈,41.28,觀望,16.51,26961,1251006.91
3,2003-09,29.0,綠燈,46.40,觀望,16.51,26961,1200050.62
4,2003-10,31.0,綠燈,44.51,觀望,16.51,26961,1294144.51
...,...,...,...,...,...,...,...,...
270,2025-12,38.0,紅燈,247.20,觀望,16.51,26961,7117720.51
271,2026-01,39.0,紅燈,264.00,觀望,16.51,26961,7716254.71
272,2026-02,40.0,紅燈,286.20,觀望,16.51,26961,8573614.51
273,2026-03,39.0,紅燈,318.00,觀望,16.51,26961,8088316.51


## 策略2: 純「定期定額法」（regular_fixed_buy）

In [88]:
# --------------------------------------------------
# 策略二：純「定期定額法」 (無需外部參數)
# --------------------------------------------------
df_res_regular = backtest_engine(
    df=df_filtered,
    initial_capital=initial_money,
    buy_strategy_func=regular_fixed_buy,
    sell_strategy_func=never_sell
)
asset_regular = df_res_regular['當下總資產'].iloc[-1]
roi_regular = ((asset_regular - initial_money) / initial_money) * 100

print("====== 策略二：純定期定額 (月扣1萬/不賣) ======")
print(f"期末總金額: NT$ {round(asset_regular):,}")
print(f"回測報酬率: {roi_regular:+.2f}%\n")


save_to_excel(df_res_regular, '2_純定期定額')


df_res_regular

====== 策略二：純定期定額 (月扣1萬/不賣) ======
期末總金額: NT$ 5,948,433
回測報酬率: +494.84%



,年月,景氣分數,燈號,0050股價,執行動作,現金水位,持股水位(股),當下總資產
0,2003-06,20.0,黃藍燈,37.10,買進 269 股,990022.79,269,1000000.00
1,2003-07,24.0,綠燈,37.09,買進 242 股,980033.03,511,1001127.11
2,2003-08,26.0,綠燈,41.28,買進 215 股,970057.03,726,1003743.43
3,2003-09,29.0,綠燈,46.40,買進 224 股,960086.79,950,1002371.29
4,2003-10,31.0,綠燈,44.51,買進 208 股,950102.79,1158,1005686.79
...,...,...,...,...,...,...,...,...
270,2025-12,38.0,紅燈,247.20,觀望,33.14,19828,5234625.14
271,2026-01,39.0,紅燈,264.00,觀望,33.14,19828,5674806.74
272,2026-02,40.0,紅燈,286.20,觀望,33.14,19828,6305337.14
273,2026-03,39.0,紅燈,318.00,觀望,33.14,19828,5948433.14


## 策略3: 不定期不定額 (無需外部參數)

In [89]:
# --------------------------------------------------
# 策略三：不定期不定額
# --------------------------------------------------
df_res_irregular = backtest_engine(
    df=df_filtered,
    initial_capital=initial_money,
    buy_strategy_func=irregular_dynamic_buy,
    sell_strategy_func=never_sell
)
asset_irregular = df_res_irregular['當下總資產'].iloc[-1]
roi_irregular = ((asset_irregular - initial_money) / initial_money) * 100

print("====== 策略三：不定期不定額 (低檔加碼/高檔觀望) ======")
print(f"期末總金額: NT$ {round(asset_irregular):,}")
print(f"回測報酬率: {roi_irregular:+.2f}%\n")



save_to_excel(df_res_irregular, '3_不定期不定額')

df_res_irregular

====== 策略三：不定期不定額 (低檔加碼/高檔觀望) ======
期末總金額: NT$ 6,183,926
回測報酬率: +518.39%



,年月,景氣分數,燈號,0050股價,執行動作,現金水位,持股水位(股),當下總資產
0,2003-06,20.0,黃藍燈,37.10,買進 404 股,985015.64,404,1000000.00
1,2003-07,24.0,綠燈,37.09,買進 242 股,975025.88,646,1001692.76
2,2003-08,26.0,綠燈,41.28,買進 215 股,965049.88,861,1005000.28
3,2003-09,29.0,綠燈,46.40,買進 224 股,955079.64,1085,1003372.99
4,2003-10,31.0,綠燈,44.51,買進 208 股,945095.64,1293,1007159.64
...,...,...,...,...,...,...,...,...
270,2025-12,38.0,紅燈,247.20,觀望,26.01,20613,5441858.01
271,2026-01,39.0,紅燈,264.00,觀望,26.01,20613,5899466.61
272,2026-02,40.0,紅燈,286.20,觀望,26.01,20613,6554960.01
273,2026-03,39.0,紅燈,318.00,觀望,26.01,20613,6183926.01


## 策略4：彼得林區+阿甘 (定期定額 + 不定期不定額加碼)

In [90]:
# --------------------------------------------------
# 策略四：彼得林區+阿甘 (定期定額 + 不定期不定額加碼)
# --------------------------------------------------
df_res_pl_dynamic = backtest_engine(
    df=df_filtered,
    initial_capital=initial_money,
    buy_strategy_func=peter_lynch_dynamic_buy,
    sell_strategy_func=never_sell
)
asset_pl_dynamic = df_res_pl_dynamic['當下總資產'].iloc[-1]
roi_pl_dynamic = ((asset_pl_dynamic - initial_money) / initial_money) * 100

print("====== 策略四：彼得林區+阿甘 ======")
print(f"期末總金額: NT$ {round(asset_pl_dynamic):,}")
print(f"回測報酬率: {roi_pl_dynamic:+.2f}%\n")






save_to_excel(df_res_pl_dynamic, '4_彼得林區+阿甘')

df_res_pl_dynamic

====== 策略四：彼得林區+阿甘 ======
期末總金額: NT$ 5,991,003
回測報酬率: +499.10%



,年月,景氣分數,燈號,0050股價,執行動作,現金水位,持股水位(股),當下總資產
0,2003-06,20.0,黃藍燈,37.10,買進 674 股,975001.34,674,1000000.00
1,2003-07,24.0,綠燈,37.09,買進 484 股,955021.82,1158,1002824.06
2,2003-08,26.0,綠燈,41.28,買進 431 股,935023.42,1589,1008753.02
3,2003-09,29.0,綠燈,46.40,買進 449 股,915038.43,2038,1005749.81
4,2003-10,31.0,綠燈,44.51,買進 416 股,895070.43,2454,1012862.43
...,...,...,...,...,...,...,...,...
270,2025-12,38.0,紅燈,247.20,觀望,3.31,19970,5272083.31
271,2026-01,39.0,紅燈,264.00,觀望,3.31,19970,5715417.31
272,2026-02,40.0,紅燈,286.20,觀望,3.31,19970,6350463.31
273,2026-03,39.0,紅燈,318.00,觀望,3.31,19970,5991003.31


## 策略5: 景氣燈號分批進出場策略（lamp_batch_buy）

In [91]:
# --------------------------------------------------
# 策略五：景氣燈號分批進出場策略
# --------------------------------------------------
lamp_params = {
    "buy_lamp_input": "黃藍燈",
    "max_buy": 10,
    "sell_lamp_input": "黃紅燈",
    "max_sell": 3,
    "buy_count": 0,      # 移入特定策略參數中
    "sell_count": 0,     # 移入特定策略參數中
    "buy_active": False, # 移入特定策略參數中
    "sell_active": False # 移入特定策略參數中
}
df_res_lamp = backtest_engine(
    df=df_filtered,
    initial_capital=initial_money,
    buy_strategy_func=lamp_batch_buy,
    sell_strategy_func=lamp_batch_sell,
    strategy_params=lamp_params
)
asset_lamp = df_res_lamp['當下總資產'].iloc[-1]
roi_lamp = ((asset_lamp - initial_money) / initial_money) * 100

print("====== 策略五：景氣燈號分批進出場 =====")
print(f"期末總金額: NT$ {round(asset_lamp):,}")
print(f"回測報酬率: {roi_lamp:+.2f}%\n")





save_to_excel(df_res_lamp, '5_景氣燈號分批進出場')

df_res_lamp

====== 策略五：景氣燈號分批進出場 =====
期末總金額: NT$ 6,780,824
回測報酬率: +578.08%



,年月,景氣分數,燈號,0050股價,執行動作,現金水位,持股水位(股),當下總資產
0,2003-06,20.0,黃藍燈,37.10,買進 2696 股,900005.36,2696,1000000.00
1,2003-07,24.0,綠燈,37.09,觀望,900005.36,2696,1011296.24
2,2003-08,26.0,綠燈,41.28,觀望,900005.36,2696,1025099.76
3,2003-09,29.0,綠燈,46.40,觀望,900005.36,2696,1020004.32
4,2003-10,31.0,綠燈,44.51,觀望,900005.36,2696,1029413.36
...,...,...,...,...,...,...,...,...
270,2025-12,38.0,紅燈,247.20,觀望,6780824.21,0,6780824.21
271,2026-01,39.0,紅燈,264.00,觀望,6780824.21,0,6780824.21
272,2026-02,40.0,紅燈,286.20,觀望,6780824.21,0,6780824.21
273,2026-03,39.0,紅燈,318.00,觀望,6780824.21,0,6780824.21


# Step5: 繪圖程式碼

## 修正2

In [92]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

def plot_subplots_backtest(df_res_agani, df_res_lamp):
    """
    使用 Plotly 繪製上下兩個子圖：
    - 子圖 1 (上): 左軸 0050 股價，右軸 總報酬率 (%)
    - 子圖 2 (下): 左軸 0050 股價，右軸 當下總資產 (元)
    兩圖共享 X 軸，可同步放大縮小。
    """
    x_timeline = df_res_agani['年月'].astype(str).tolist()

    # 從資料第一筆自動逆推初始資金
    init_capital = df_res_agani['當下總資產'].iloc[0]

    # 計算總報酬率 (%)
    roi_agani = (df_res_agani['當下總資產'] - init_capital) / init_capital * 100
    roi_lamp = (df_res_lamp['當下總資產'] - init_capital) / init_capital * 100

    # 提取絕對資產與股價
    asset_agani = df_res_agani['當下總資產']
    asset_lamp = df_res_lamp['當下總資產']
    price_0050 = df_res_agani['0050股價']

    # 1. 建立 2 橫列、1 直欄的子圖，並設定兩者都有雙 Y 軸 (secondary_y=True)
    #    shared_xaxes=True 讓上下兩張圖的時間軸同步縮放
    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.1, # 上下圖之間的間距
        subplot_titles=(
            "【子圖一】0050 還原股價 vs. 總報酬率 (%)",
            "【子圖二】0050 還原股價 vs. 當下總資產 (元)"
        ),
        specs=[[{"secondary_y": True}], [{"secondary_y": True}]]
    )

    # 景氣燈號標記樣式設定
    lamp_config = {
        '紅燈':    {'color': '#E32626', 'symbol': 'circle', 'size': 11},
        '黃紅燈':  {'color': '#F3C543', 'symbol': 'circle', 'size': 9},
        '綠燈':    {'color': '#2AA851', 'symbol': 'triangle-up', 'size': 7},
        '黃藍燈':  {'color': '#8B70F7', 'symbol': 'circle', 'size': 10},
        '藍燈':    {'color': '#3350F2', 'symbol': 'triangle-down', 'size': 7}
    }

    # =========================================================================
    #  【第 1 排：子圖一】左軸：股價 / 右軸：總報酬率
    # =========================================================================
    # 左軸：0050 還原股價
    fig.add_trace(
        go.Scatter(x=x_timeline, y=price_0050, name="0050 還原股價(元)",
                   mode="lines+markers", line=dict(color="black", width=1.2, dash="dash"),
                   marker=dict(symbol="square", size=3), showlegend=True),
        row=1, col=1, secondary_y=False
    )
    # 右軸：阿甘總報酬率
    fig.add_trace(
        go.Scatter(x=x_timeline, y=roi_agani, name="阿甘投資法：總報酬率(%)",
                   mode="lines+markers", line=dict(color="#7B8CEF", width=2), marker=dict(size=3)),
        row=1, col=1, secondary_y=True
    )
    # 右軸：景氣燈號總報酬率
    fig.add_trace(
        go.Scatter(x=x_timeline, y=roi_lamp, name="景氣燈號：總報酬率(%)",
                   mode="lines", line=dict(color="#E61A8A", width=2)),
        row=1, col=1, secondary_y=True
    )
    # 右軸疊加：景氣燈號點 (報酬率)
    for lamp_name, cfg in lamp_config.items():
        mask = df_res_lamp['燈號'] == lamp_name
        if mask.any():
            fig.add_trace(
                go.Scatter(x=df_res_lamp[mask]['年月'], y=roi_lamp[mask], name=f"燈號：{lamp_name}",
                           mode="markers", marker=dict(color=cfg['color'], symbol=cfg['symbol'], size=cfg['size'], line=dict(color='black', width=1)),
                           showlegend=False), # 隱藏重複的圖例說明
                row=1, col=1, secondary_y=True
            )

    # =========================================================================
    #  【第 2 排：子圖二】左軸：股價 / 右軸：當下總資產
    # =========================================================================
    # 左軸：0050 還原股價 (與上圖相同，重繪於下圖左軸)
    fig.add_trace(
        go.Scatter(x=x_timeline, y=price_0050, name="0050 還原股價(元)",
                   mode="lines+markers", line=dict(color="black", width=1.2, dash="dash"),
                   marker=dict(symbol="square", size=3), showlegend=False), # 圖例上圖有了，這裡隱藏
        row=2, col=1, secondary_y=False
    )
    # 右軸：阿甘當下總資產
    fig.add_trace(
        go.Scatter(x=x_timeline, y=asset_agani, name="阿甘投資法：當下總資產",
                   mode="lines+markers", line=dict(color="#4A6572", width=2), marker=dict(size=3)),
        row=2, col=1, secondary_y=True
    )
    # 右軸：景氣燈號當下總資產
    fig.add_trace(
        go.Scatter(x=x_timeline, y=asset_lamp, name="景氣燈號：當下總資產",
                   mode="lines", line=dict(color="#FF7043", width=2)),
        row=2, col=1, secondary_y=True
    )
    # 右軸疊加：景氣燈號點 (總資產)
    for lamp_name, cfg in lamp_config.items():
        mask = df_res_lamp['燈號'] == lamp_name
        if mask.any():
            fig.add_trace(
                go.Scatter(x=df_res_lamp[mask]['年月'], y=asset_lamp[mask],
                           mode="markers", marker=dict(color=cfg['color'], symbol=cfg['symbol'], size=cfg['size'], line=dict(color='black', width=1)),
                           showlegend=False),
                row=2, col=1, secondary_y=True
            )

    # =========================================================================
    #  全域版面優化與細節設定 (Layout)
    # =========================================================================
    fig.update_layout(
        title=dict(text="策略績效雙視角回測分析 (Subplots)", font=dict(size=20)),
        hovermode="x unified",  # 滑鼠移到該月份，上下兩張圖會同時觸發提示框
        legend=dict(x=1.12, y=1, bgcolor="rgba(255,255,255,0.8)"), # 將圖例移到外側避免擋到圖
        width=1300,
        height=900
    )

    # 軸標題設定 - 子圖一 (Row 1)
    fig.update_yaxes(title_text="0050 股價(元)", row=1, col=1, secondary_y=False)
    fig.update_yaxes(title_text="總報酬率 (%)", tickformat=".0f", ticksuffix="%", row=1, col=1, secondary_y=True)

    # 軸標題設定 - 子圖二 (Row 2)
    fig.update_yaxes(title_text="0050 股價(元)", row=2, col=1, secondary_y=False)
    fig.update_yaxes(title_text="當下總資產 (元)", tickformat=",.0f", row=2, col=1, secondary_y=True) # 加上千分位逗號

    # X 軸與動態放大拉條設定 (套用在 Row 2 底部)
    fig.update_xaxes(title_text="年/月", row=2, col=1)
    fig.update_xaxes(
        type="category",
        rangeslider=dict(visible=True, thickness=0.04), # 底部時間拉條
    )

    fig.show()
    fig.write_html("verse.html")

# --- 呼叫範例 ---
plot_subplots_backtest(df_res_all_in, df_res_lamp)

